# Runtime-scaling scan planner

Generates a midpoint-convention BubbleMaster GPU scan from three physical
inputs: the gamma_* range and the kR_* range and N_omega. Grid spacing
(`dgamma_ij=0.4`, `dt=2`) and the resolution/numerics (`wall_points=20`,
`N_min=128`, `panels_per_oscillation=40`, `how_often_ds=5`,
`s_batch_size=128`) are locked to the values validated in
`14_high_gamma_convergence_check.ipynb` -- not exposed as scan inputs here.

Re-running this notebook with a different `(GAMMA_STAR_MIN, GAMMA_STAR_MAX,
KR_MIN, KR_MAX, N_OMEGA)` writes a separate, distinctly-named scan under
`data/runtime_scan_.../` and `scripts/runtime_scan_...sh` -- so several scans
(e.g. increasing `GAMMA_STAR_MAX`) can be run and timed side by side to see
how total runtime scales with the domain size.

In [1]:
from pathlib import Path
from types import SimpleNamespace

import h5py
import numpy as np
from scipy.optimize import brentq

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams, write_2d_setup, collision_wall_width
from ptbuilder.weight_scan_diagnostics import load_kinematics

ROOT = Path(pt.__file__).resolve().parent.parent
LAMBDA_BAR = 0.84

## Scan inputs

Only these three define the physical scan domain.

In [2]:
GAMMA_STAR_MIN = 1.0
GAMMA_STAR_MAX = 9.6
KR_MIN = 1.0
KR_MAX = 100.0
N_OMEGA = 32

# Optional overrides -- production defaults, only change these deliberately.
PAIR_GAMMA_FACTOR = 2.5   # gamma_ij,max = PAIR_GAMMA_FACTOR * GAMMA_STAR_MAX
TIME_FACTOR       = 1.4   # T_MAX = TIME_FACTOR * R_*(GAMMA_STAR_MAX)

## Locked grid spacing and numerics

Not scan inputs -- these are the production values validated elsewhere in
this repo (`dgamma_ij`/`dt` in `13_reconstruction_scan_resolution.ipynb`;
`N_min`/`panels_per_oscillation`/`wall_points`/`how_often_ds` up to
gamma_ij=24 in `14_high_gamma_convergence_check.ipynb`).

In [3]:
DGAMMA_IJ = 0.4
DT        = 2.0
N_K       = 51

LOCKED = dict(n_min=128, ppw=40, wall_points=20, how_often_ds=5, s_batch_size=128)
CUTOFF_TYPE = 0
BABY_STEPS  = 100
GPU_ACCOUNT = 'm3166_g'

In [4]:
if not (0 < KR_MIN < KR_MAX):
    raise ValueError('Require 0 < KR_MIN < KR_MAX')
if not (1.0 <= GAMMA_STAR_MIN <= GAMMA_STAR_MAX):
    raise ValueError('Require 1 <= GAMMA_STAR_MIN <= GAMMA_STAR_MAX')
if N_OMEGA < 2:
    raise ValueError('N_OMEGA must be >= 2')

instanton_path = ROOT / f'data/phi4_lambda_bar{LAMBDA_BAR:g}/instanton.h5'
kin = load_kinematics(instanton_path)

class CachedPhi4Potential:
    def __init__(self, lambda_bar):
        self.params = {'lambda_bar': float(lambda_bar)}
    def to_hdf5(self, group):
        group.attrs.create('type', 'phi4', dtype=h5py.string_dtype(encoding='ascii'))
        group.attrs['phi_false'] = 0.0
        group.attrs['phi_true']  = 1.0
        group.attrs['lambda_bar'] = self.params['lambda_bar']

model = SimpleNamespace(instanton=kin.profile, kinematics=kin,
                        potential=CachedPhi4Potential(LAMBDA_BAR))

def time_at_gamma(gamma):
    if gamma <= 1.0:
        return 0.0
    return brentq(lambda t: float(kin.Gamma(t)) - gamma, 0.0, 1.0e7)

def rstar_at_gamma(gamma):
    t = time_at_gamma(gamma)
    return 2.0 * float(kin.R(t, r='mid'))

GAMMA_IJ_MAX = PAIR_GAMMA_FACTOR * GAMMA_STAR_MAX
R_STAR_MAX   = rstar_at_gamma(GAMMA_STAR_MAX)
T_MAX        = TIME_FACTOR * R_STAR_MAX
OMEGA_MIN    = KR_MIN / R_STAR_MAX

n_gamma = int(np.ceil((GAMMA_IJ_MAX - GAMMA_STAR_MIN) / DGAMMA_IJ)) + 1
gamma_ij_grid = np.linspace(GAMMA_STAR_MIN, GAMMA_IJ_MAX, n_gamma)

print(f'gamma_ij: {gamma_ij_grid[0]:.3f} .. {gamma_ij_grid[-1]:.3f}  ({n_gamma} pairs, step {DGAMMA_IJ})')
print(f'R_*(gamma_star_max={GAMMA_STAR_MAX}) = {R_STAR_MAX:.3f}')
print(f'T_MAX = {T_MAX:.3f}   OMEGA_MIN = {OMEGA_MIN:.5f}')

gamma_ij: 1.000 .. 24.000  (59 pairs, step 0.4)
R_*(gamma_star_max=9.6) = 133.145
T_MAX = 186.402   OMEGA_MIN = 0.00751


## Write setups + manifest

Folder name encodes the scan's physical inputs, so different
`(GAMMA_STAR_MIN, GAMMA_STAR_MAX, KR_MIN, KR_MAX, N_OMEGA)` choices land in
separate directories automatically.

In [5]:
tag = (f'lb{LAMBDA_BAR:g}_gs{GAMMA_STAR_MIN:g}-{GAMMA_STAR_MAX:g}'
       f'_kR{KR_MIN:g}-{KR_MAX:g}_nw{N_OMEGA}')
scan_root   = ROOT / 'data' / f'runtime_scan_{tag}'
setup_dir   = scan_root / 'setups'
output_root = scan_root / 'gpu'
setup_dir.mkdir(parents=True, exist_ok=True)
output_root.mkdir(parents=True, exist_ok=True)
(ROOT / 'logs').mkdir(exist_ok=True)

params = BubbleMasterParams(
    dz=None, wall_points=LOCKED['wall_points'],
    n_w=N_OMEGA, n_k=N_K, how_often_ds=LOCKED['how_often_ds'],
    baby_steps=BABY_STEPS, cutoff_type=CUTOFF_TYPE, t_0_scal=1.0,
    collision_radius='mid',
)

manifest_rows = []
for index, gamma_ij in enumerate(gamma_ij_grid):
    t_first = time_at_gamma(gamma_ij)
    if t_first >= T_MAX:
        raise RuntimeError(f'Collision time exceeds T_MAX at gamma_ij={gamma_ij}')
    n_t = int(np.ceil((T_MAX - t_first) / DT)) + 1
    times = np.linspace(t_first, T_MAX, n_t)

    smallest_target = max(GAMMA_STAR_MIN, gamma_ij / PAIR_GAMMA_FACTOR)
    omega_max = KR_MAX / rstar_at_gamma(smallest_target)
    if omega_max <= OMEGA_MIN:
        raise RuntimeError(f'Empty omega range at gamma_ij={gamma_ij}')
    omega = np.geomspace(OMEGA_MIN, omega_max, N_OMEGA)

    name = f'g{gamma_ij:08.3f}'.replace('.', 'p')
    setup_path = setup_dir / f'{name}.h5'
    write_2d_setup(model, gamma_ij, times, setup_path, params, wlist=omega)
    with h5py.File(setup_path, 'a') as handle:
        handle.attrs['gamma_star_min']    = GAMMA_STAR_MIN
        handle.attrs['gamma_star_max']    = GAMMA_STAR_MAX
        handle.attrs['kR_min']            = KR_MIN
        handle.attrs['kR_max']            = KR_MAX
        handle.attrs['pair_gamma_factor'] = PAIR_GAMMA_FACTOR
        handle.attrs['time_factor']       = TIME_FACTOR
        handle.attrs['R_star_max']        = R_STAR_MAX

    manifest_rows.append((index, gamma_ij, n_t, setup_path.relative_to(ROOT), name))

manifest = scan_root / 'manifest.tsv'
manifest.write_text(
    'index\tgamma_ij\tn_t\tsetup\tname\n' +
    ''.join(f'{i}\t{g:.12g}\t{nt}\t{s}\t{n}\n' for i, g, nt, s, n in manifest_rows)
)
print(f'Wrote {len(manifest_rows)} setups under {setup_dir.relative_to(ROOT)}')
print(f'Manifest: {manifest.relative_to(ROOT)}')

Wrote 59 setups under data/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32/setups
Manifest: data/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32/manifest.tsv


## Rough walltime estimate

Extrapolates from the two real `gamma_ij=24` timings in
`14_high_gamma_convergence_check.ipynb` (`n_s*n_z`-proportional streaming
cost). This is a single-regime calibration point, not a robust fit -- treat
it as a lower-bound sanity check on `GPU_WALLTIME` below, not a guarantee.
Only the largest `gamma_ij` in the grid matters, since all pairs run
concurrently on separate GPUs.

In [6]:
from ptbuilder.ic import _spatial_step, _two_bubble_ic

_A_SECONDS_PER_NSNZ = (74.06 * 60) / (246104 * 560858)   # calibrated at gamma_ij=24
_SAFETY_FACTOR = 3.0   # this model is a single anchor point -- pad generously

g_max = gamma_ij_grid[-1]
dz_target = _spatial_step(model.instanton, g_max, params)
z, _, d, ds = _two_bubble_ic(model.instanton, g_max, dz_target, 'mid')
n_z_max = len(z)
n_s_max = round(1.2 * d / ds)
est_seconds = _A_SECONDS_PER_NSNZ * n_s_max * n_z_max * _SAFETY_FACTOR

print(f'largest gamma_ij in grid: {g_max:.3f}  (n_s={n_s_max:,}  n_z={n_z_max:,})')
print(f'estimated walltime (with {_SAFETY_FACTOR}x safety margin): '
      f'{est_seconds/3600:.2f} h  -- set GPU_WALLTIME below accordingly')

largest gamma_ij in grid: 24.000  (n_s=560,858  n_z=246,104)
estimated walltime (with 3.0x safety margin): 3.70 h  -- set GPU_WALLTIME below accordingly


## SLURM script

Mirrors `09_adaptive_bm_scan.ipynb`'s GPU job: one exclusive task per
manifest row, four per node, all launched concurrently in the background.

In [7]:
GPU_WALLTIME = '04:00:00'   # override manually if the estimate above says otherwise

gpus_per_node = 4
n_tasks = len(manifest_rows)
n_nodes = (n_tasks + gpus_per_node - 1) // gpus_per_node
relative_manifest = manifest.relative_to(ROOT)
relative_output   = output_root.relative_to(ROOT)

script_path = ROOT / 'scripts' / f'runtime_scan_{tag}_gpu.sh'
script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{tag}_gpu
#SBATCH --output=logs/bm_{tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {GPU_ACCOUNT}
#SBATCH --nodes={n_nodes}
#SBATCH --ntasks={n_tasks}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={GPU_WALLTIME}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"
pids=()
while IFS=$'\t' read -r index gamma n_t setup name; do
    [[ "$index" == "index" ]] && continue
    output="{relative_output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {LOCKED['n_min']} --filon-panels-per-osc {LOCKED['ppw']} --s-batch-size {LOCKED['s_batch_size']} &
    pids+=("$!")
done < {relative_manifest}
status=0
for pid in "${{pids[@]}}"; do
    wait "$pid" || status=1
done
exit "$status"
''')
script_path.chmod(0o755)
print(f'Written: {script_path.relative_to(ROOT)}  ({n_tasks} tasks, {n_nodes} nodes)')
print(f'Submit: sbatch {script_path.relative_to(ROOT)}')

Written: scripts/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32_gpu.sh  (59 tasks, 15 nodes)
Submit: sbatch scripts/runtime_scan_lb0.84_gs1-9.6_kR1-100_nw32_gpu.sh
